# 04 — Reasoning Walkthrough

> **SYNTHETIC DATA DEMO** — All telemetry and fault scenarios in this notebook are programmatically generated. Results demonstrate architecture and controlled functional behaviour, not real-world accuracy or statistical validation.

Runs the eleven final canonical demo questions through the same conversational entry point. Without an API key, the deterministic fallback uses the approved analytics; with `ANTHROPIC_API_KEY`, the Claude tool-use loop is used.


In [1]:
# Colab/local bootstrap: discover the repository, clone only when needed.
from pathlib import Path
import os, sys, subprocess

start = Path.cwd().resolve()
candidates = [start, *start.parents]
ROOT = next((p for p in candidates if (p / "src").exists() and (p / "config").exists()), None)
if ROOT is None:
    if Path("/content").exists():
        repo = Path("/content/intelligent-data-logger-demo")
        if not repo.exists():
            subprocess.run(["git", "clone", "https://github.com/Engr-Daniel/intelligent-data-logger-demo.git", str(repo)], check=True)
        ROOT = repo
    else:
        raise RuntimeError("Could not locate the repository. Open the notebook from the cloned repo or use Colab.")
os.chdir(ROOT)
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

Repository root: /mnt/data/m6fix/intelligent-data-logger-demo


In [2]:
from src.interface.status_card import current_status
from pprint import pprint
pprint(current_status())

{'active_alarm': None,
 'battery_soc_pct': 30.0,
 'estimated_battery_runway_hours': 2.53,
 'grid_export_w': 0.0,
 'grid_import_w': 712.0,
 'historical_alarm_events': 3,
 'inverter_state': 'normal',
 'load_w': 712.0,
 'pv_generation_w': 0.0,
 'synthetic_data': True,
 'timestamp': '2026-11-28 23:55:00+01:00'}


In [3]:
from src.reasoning.agent import answer
queries = [
    "What's the current state of my system?",
    'Why did our energy production drop yesterday?',
    'How much of our consumption came from solar this month?',
    "What's unusual about the inverter's behaviour?",
    'The customer says the inverter just failed on its own — what actually happened around that time?',
    'How long will my battery last tonight at current usage?',
    'How sustainable was our energy usage this month?',
    "What's our ROI so far?",
    'Do we have enough data to tell what happened on 2026-09-18?',
    'Was the customer definitely responsible for this inverter failure?',
    'What happened when the grid went down on 2026-10-10?',
]
for i, q in enumerate(queries, 1):
    print(f"\nQ{i}: {q}")
    print("A:", answer(q))



Q1: What's the current state of my system?


A: Current system status was read from the latest complete telemetry interval. Evidence: pv_generation_w=0.0, load_w=712.0, battery_soc_pct=30.0, estimated_battery_runway_hours=2.53.

Q2: Why did our energy production drop yesterday?


A: The strongest supported explanation is lower solar irradiance. PV production was materially lower and the drop closely tracked lower irradiance, with no inverter alarm or abnormal operating state. Evidence: target_mean_pv_power_w=1685.9, baseline_mean_pv_power_w=4327.9, pv_drop_pct_vs_surrounding_days=61.0, target_mean_irradiance_wm2=272.5.

Q3: How much of our consumption came from solar this month?


A: Energy balance for the latest 30 days was calculated from stored telemetry. Evidence: pv_generation_kwh=1096.61, consumption_kwh=620.09, requested_consumption_kwh=620.09, unmet_load_kwh=0.0.

Q4: What's unusual about the inverter's behaviour?


A: Inverter alarm/derating was observed with sustained load above the inverter rating in the alarm window. Evidence: alarm_rows=3, abnormal_state_rows=15, rows_above_inverter_rating_in_event_window=8, duration_above_rating_min=40.0. Observed irradiance- and temperature-normalized PV performance shows a sustained downward multi-week trend. Evidence: early_period_median_performance_ratio=0.965, late_period_median_performance_ratio=0.8878, normalized_performance_decline_pct=8.0, daily_trend_slope=-0.0007942.

Q5: The customer says the inverter just failed on its own — what actually happened around that time?


...

A: Sustainability metrics for the latest 30 days were calculated using the configured grid-displacement assumption. Evidence: renewable_fraction_pct=73.7, estimated_avoided_emissions_kgco2e=205.62, solar_self_consumed_kwh=456.94.

Q8: What's our ROI so far?


A: Simple financial metrics were calculated from synthetic telemetry and configured tariff/CAPEX assumptions. Evidence: avoided_energy_cost=416571.75, net_savings=416571.75, simple_roi_pct=6.41, annualized_net_savings=1267072.41.

Q9: Do we have enough data to tell what happened on 2026-09-18?


A: Telemetry is unavailable from 2026-09-18T12:00:00+01:00 to 2026-09-18T12:45:00+01:00; causal diagnosis should abstain for that interval.

Q10: Was the customer definitely responsible for this inverter failure?


A: Sustained overload preceded the inverter alarm. Evidence: peak_load_w=6403.9, inverter_rating_w=5000, duration_above_rating_min=40.0, peak_battery_discharge_w=4000.0. The telemetry supports a leading technical explanation, but it cannot prove customer responsibility, intent, or warranty liability.

Q11: What happened when the grid went down on 2026-10-10?


A: Utility grid loss was followed by islanded backup operation; local PV/battery supply maintained the requested load until grid restoration. Evidence: outage_start=2026-10-10T21:35:00+01:00, outage_end=2026-10-10T22:35:00+01:00, outage_duration_minutes=60.0, mean_pre_event_grid_import_w=66.6.


The adversarial responsibility question is intentionally calibrated: telemetry may support an overload diagnosis, but it cannot prove customer responsibility, intent, or warranty liability. The grid-outage question is separately grounded in observed grid availability, inverter state, battery response, local power balance, and restoration evidence.
